# ASAP7y SNR using a Hao et al.-style voltage-imaging metric

This notebook estimates ROI-level ASAP7y SNR using the same basic logic used in Hao et al. 2024 for GEVI recordings:

1. Correct slow baseline / photobleaching with a local piece-wise baseline.
2. Detect voltage-associated optical events.
3. Remove detected events from the trace when estimating noise.
4. Estimate noise as the standard deviation of the **20 Hz high-pass filtered, event-removed trace**.
5. Define signal as detected event peak amplitude in ΔF/F.
6. Define SNR as:

\[
\mathrm{SNR}_{Hao-like} = \frac{\mathrm{peak\ event\ amplitude}}{\sigma_{20Hz\ highpass,\ event-removed}}
\]

This is not intended to claim that dendritic ASAP7y events are the same as ASAP5 somatic APs or mEPSPs. It is a practical adaptation of the published GEVI SNR convention to your full-session dendritic dF/F traces.

Important implementation choice: **plotting cells are standalone**. There are helper functions for loading and computing the metric, but there are no custom plotting functions, so each plot can be edited directly in the cell that produces it.

In [ ]:
import os
import sys
import glob
import json
import h5py
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

from scipy.signal import butter, sosfiltfilt, find_peaks
from scipy.ndimage import uniform_filter1d
from scipy.stats import mannwhitneyu

try:
    from statsmodels.stats.multitest import multipletests
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.extraction import load_voltage_roi_transform_h5
from vip_slap2_analysis.voltage import analysis
from vip_slap2_analysis.common.qc import robust_sigma
from vip_slap2_analysis.plotting.plot_psth import plot_voltage_mean_image_response_heatmap
from vip_slap2_analysis.utils.utils import save_figure

import matplotlib.pyplot as plt
from IPython.display import display, HTML

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

display(HTML("<style>.container { width:100% !important; }</style>"))
warnings.filterwarnings("default")

In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib notebook

## Build session registry

In [ ]:
today_str = datetime.today().strftime("%Y-%m-%d")

BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
SAVE_PATH = Path(
    r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures\voltage_plots"
)

TARGET_MICE = [
    826031,
    826032,
]

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]

# Voltage extraction writes files like:
#   voltage_session_traces_dff_robust_f0_trial.h5
TRACE_VARIANT = "dff_robust_f0_trial"
SIGNAL = "dff"      # one of: "raw_f", "f0", "dff"

# Optional direct override. Leave as None to resolve from asset.derived_dir / "voltage".
SESSION_TRACE_H5 = None

SAVE_PATH.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {SAVE_PATH}")

In [ ]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

process_df = registry.sessions(
    subject_ids=TARGET_MICE,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
    paradigms=PARADIGMS,
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Found {len(assets)} candidate sessions.")
display(process_df)

## Analysis parameters

These are the parameters most likely to edit. The defaults are intentionally close to the GEVI SNR method in Hao et al.:

- `BASELINE_WINDOW_SEC = 0.5`: local piece-wise baseline window.
- `BASELINE_PERCENTILES = (30, 80)`: average the middle portion of each window to avoid spikes/extreme values.
- `NOISE_HIGHPASS_HZ = 20`: estimate noise from a 20 Hz high-pass filtered trace after removing detected events.
- `SIGNAL_SUMMARY = "median"`: summarize event peak SNR per ROI using the median detected-event SNR.

Because your traces are dendritic and session-long, the pooled analysis uses distributed windows instead of reading entire sessions into memory.

In [ ]:
# Example-session display
EXAMPLE_ASSET_INDEX = 0
TRACE_START_SEC = None       # None -> begin 30 s after trace start
TRACE_DURATION_SEC = 15

# Analysis rate
# Hao et al. often analyze camera-rate data at ~400-500 Hz for soma/mEPSP imaging.
# Here we downsample the ~10.8 kHz SLAP2 dF/F trace to 1000 Hz by default, preserving fast transients.
ANALYSIS_RATE_HZ = 1000.0    # set None to keep native sampling

# Hao-like baseline and noise estimation
BASELINE_WINDOW_SEC = 0.5
BASELINE_PERCENTILES = (30, 80)
NOISE_HIGHPASS_HZ = 20.0

# Event detection for signal peaks
EVENT_POLARITY = "positive"  # "positive" for inverted ASAP dF/F; use "negative" if needed
EVENT_THRESHOLD_SD = 3     # threshold relative to initial 20 Hz high-pass noise estimate
EVENT_PROMINENCE_SD = 2.0
MIN_PEAK_DISTANCE_MS = 15.0
EVENT_MASK_PRE_MS = 10.0
EVENT_MASK_POST_MS = 20.0

# Pooled all-session metrics
METRIC_WINDOW_SEC = 20.0
N_METRIC_WINDOWS = 40
MIN_VALID_SAMPLES = 500
MIN_EVENTS_FOR_VALID_ROI = 5
MIN_VALID_WINDOWS = 3
SIGNAL_SUMMARY = "median"   # "median", "mean", or "p90"

# Plotting/output
SNR_METRIC = "snr_peak_median"
HIST_BINS = 35
SCATTER_ALPHA = 0.55
SAVE_FIGURES = True
SAVE_TABLE = True

print("Hao-like SNR metric:")
print(f"  baseline: piece-wise {BASELINE_WINDOW_SEC}s, average of {BASELINE_PERCENTILES[0]}-{BASELINE_PERCENTILES[1]} percentile values")
print(f"  noise: SD of {NOISE_HIGHPASS_HZ:g} Hz high-pass filtered, event-removed trace")
print(f"  signal: detected event peak amplitude in baseline-corrected dF/F")

## Helper functions: data loading and metric computation only

The functions below do not make plots. They are kept here to avoid repeating HDF5 slicing, downsampling, baseline correction, and SNR calculation code throughout the notebook.

In [ ]:
def resolve_trace_h5(asset, trace_variant=TRACE_VARIANT, override=SESSION_TRACE_H5):
    """Resolve the session-long voltage trace file while retaining the asset scheme."""
    if override is not None:
        path = Path(override)
    else:
        path = Path(asset.derived_dir) / "voltage" / f"voltage_session_traces_{trace_variant}.h5"

    if path.exists():
        return path

    candidates = sorted((Path(asset.derived_dir) / "voltage").glob("voltage_session_traces*.h5"))
    if candidates:
        warnings.warn(f"Could not find expected file {path.name}; using {candidates[0].name}")
        return candidates[0]

    raise FileNotFoundError(f"No session trace H5 found for asset {getattr(asset, 'session_id', asset)} at {path}")


def _decode_array(values):
    out = []
    for v in values:
        if isinstance(v, bytes):
            out.append(v.decode())
        else:
            out.append(str(v))
    return out


def get_asset_session_id(asset):
    for key in ["session_id", "session", "name"]:
        if hasattr(asset, key):
            return getattr(asset, key)
    try:
        return asset.metadata.get("session_id", "unknown_session")
    except Exception:
        return "unknown_session"


def get_asset_subject_id(asset):
    try:
        for key in ["subject_id", "mouse_id", "mouse"]:
            if key in asset.metadata:
                return asset.metadata[key]
    except Exception:
        pass
    return np.nan


def get_asset_depth(asset, dmd):
    try:
        key = f"{str(dmd).lower()}_depth"
        if key in asset.metadata:
            return asset.metadata[key]
        key = f"{str(dmd).upper().lower()}_depth"
        if key in asset.metadata:
            return asset.metadata[key]
        if dmd == "DMD1" and "dmd1_depth" in asset.metadata:
            return asset.metadata["dmd1_depth"]
        if dmd == "DMD2" and "dmd2_depth" in asset.metadata:
            return asset.metadata["dmd2_depth"]
    except Exception:
        pass
    return np.nan


def get_dmd_info(h5_path, dmd, signal=SIGNAL):
    with h5py.File(h5_path, "r") as h5:
        g = h5[dmd]
        t = g["timebase_sec"][:]
        ds = g[signal]
        if ds.shape[0] == len(t):
            orientation = "time_by_roi"
            n_time, n_rois = ds.shape
        elif ds.shape[-1] == len(t):
            orientation = "roi_by_time"
            n_rois, n_time = ds.shape
        else:
            raise ValueError(f"Could not infer orientation for {h5_path} {dmd}/{signal}, shape={ds.shape}, len(t)={len(t)}")
        if "roi_ids" in g:
            roi_ids = _decode_array(g["roi_ids"][:])
        else:
            roi_ids = [f"{dmd}_roi{i:04d}" for i in range(n_rois)]
    return {
        "t_start": float(t[0]),
        "t_stop": float(t[-1]),
        "dt_median": float(np.nanmedian(np.diff(t))),
        "fs": float(1.0 / np.nanmedian(np.diff(t))),
        "n_time": int(n_time),
        "n_rois": int(n_rois),
        "roi_ids": roi_ids,
        "orientation": orientation,
    }


def read_dmd_segment_all_rois(h5_path, dmd, t0, t1, signal=SIGNAL):
    """Read a time segment for all ROIs as an array shaped (n_rois, n_samples)."""
    with h5py.File(h5_path, "r") as h5:
        g = h5[dmd]
        t = g["timebase_sec"][:]
        i0 = int(np.searchsorted(t, t0, side="left"))
        i1 = int(np.searchsorted(t, t1, side="right"))
        i0 = max(0, min(i0, len(t) - 1))
        i1 = max(i0 + 1, min(i1, len(t)))
        t_seg = t[i0:i1]
        ds = g[signal]
        if ds.shape[0] == len(t):
            X = ds[i0:i1, :].T
        elif ds.shape[-1] == len(t):
            X = ds[:, i0:i1]
        else:
            raise ValueError(f"Could not infer orientation for {h5_path} {dmd}/{signal}")
        X = np.asarray(X, dtype=np.float32)
    return t_seg, X


def block_average_trace(x, t, target_rate_hz):
    if target_rate_hz is None:
        return np.asarray(x, dtype=float), np.asarray(t, dtype=float)
    x = np.asarray(x, dtype=float)
    t = np.asarray(t, dtype=float)
    if len(t) < 2:
        return x, t
    fs = 1.0 / np.nanmedian(np.diff(t))
    factor = int(max(1, round(fs / float(target_rate_hz))))
    if factor <= 1:
        return x, t
    n = (len(x) // factor) * factor
    if n < factor:
        return x, t
    xb = np.nanmean(x[:n].reshape(-1, factor), axis=1)
    tb = np.nanmean(t[:n].reshape(-1, factor), axis=1)
    return xb, tb


def interpolate_nans(x):
    x = np.asarray(x, dtype=float)
    finite = np.isfinite(x)
    if finite.all():
        return x
    if finite.sum() < 2:
        return np.full_like(x, np.nan)
    idx = np.arange(x.size)
    y = x.copy()
    y[~finite] = np.interp(idx[~finite], idx[finite], x[finite])
    return y


def piecewise_percentile_baseline(x, fs, window_sec=BASELINE_WINDOW_SEC, percentiles=BASELINE_PERCENTILES):
    """Hao-like local baseline: mean of middle percentile values in each window, interpolated over time."""
    x = interpolate_nans(x)
    n = len(x)
    win = max(3, int(round(window_sec * fs)))
    centers = []
    values = []
    plo, phi = percentiles

    for i0 in range(0, n, win):
        i1 = min(n, i0 + win)
        w = x[i0:i1]
        w = w[np.isfinite(w)]
        if w.size == 0:
            continue
        lo, hi = np.nanpercentile(w, [plo, phi])
        mid = w[(w >= lo) & (w <= hi)]
        if mid.size == 0:
            b = np.nanmedian(w)
        else:
            b = np.nanmean(mid)
        centers.append((i0 + i1 - 1) / 2.0)
        values.append(b)

    if len(values) == 0:
        return np.full(n, np.nan)
    if len(values) == 1:
        return np.full(n, values[0], dtype=float)

    idx = np.arange(n)
    baseline = np.interp(idx, np.asarray(centers), np.asarray(values))
    return baseline


def highpass_filter(x, fs, cutoff_hz=NOISE_HIGHPASS_HZ, order=3):
    x = interpolate_nans(x)
    if not np.isfinite(x).any():
        return np.full_like(x, np.nan)
    nyq = 0.5 * fs
    cutoff = float(cutoff_hz) / nyq
    if cutoff <= 0 or cutoff >= 0.95 or len(x) < 5 * fs / max(cutoff_hz, 1):
        # Fall back to subtracting a smooth trend if the segment is too short for stable filtering.
        smooth_n = max(3, int(round(fs / max(cutoff_hz, 1))))
        trend = uniform_filter1d(x, size=smooth_n, mode="nearest")
        return x - trend
    sos = butter(order, cutoff, btype="highpass", output="sos")
    try:
        return sosfiltfilt(sos, x)
    except ValueError:
        smooth_n = max(3, int(round(fs / max(cutoff_hz, 1))))
        trend = uniform_filter1d(x, size=smooth_n, mode="nearest")
        return x - trend


def robust_std(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    return 1.4826 * mad


def compute_hao_like_snr(x, t):
    """Compute Hao-like detected-event SNR for one ROI trace segment."""
    x, t = block_average_trace(x, t, ANALYSIS_RATE_HZ)
    finite = np.isfinite(x) & np.isfinite(t)
    x = x[finite]
    t = t[finite]
    if len(x) < MIN_VALID_SAMPLES or len(t) < MIN_VALID_SAMPLES:
        return {
            "valid_window": False,
            "n_events": 0,
            "noise_sigma_20hz_hp_dff": np.nan,
            "peak_amp_median_dff": np.nan,
            "peak_amp_mean_dff": np.nan,
            "peak_amp_p90_dff": np.nan,
            "snr_peak_median": np.nan,
            "snr_peak_mean": np.nan,
            "snr_peak_p90": np.nan,
            "event_rate_hz": np.nan,
            "event_times_sec": np.array([], dtype=float),
            "event_peak_amplitudes": np.array([], dtype=float),
            "trace_corrected": np.asarray([], dtype=float),
            "trace_time_sec": np.asarray([], dtype=float),
            "event_mask": np.asarray([], dtype=bool),
        }

    fs = 1.0 / np.nanmedian(np.diff(t))
    baseline = piecewise_percentile_baseline(x, fs)
    y = x - baseline

    if EVENT_POLARITY == "negative":
        y_det = -y
    elif EVENT_POLARITY == "positive":
        y_det = y
    else:
        y_det = np.abs(y)

    hp_initial = highpass_filter(y_det, fs, NOISE_HIGHPASS_HZ)
    sigma_initial = robust_std(hp_initial)
    if not np.isfinite(sigma_initial) or sigma_initial <= 0:
        sigma_initial = np.nanstd(hp_initial)
    if not np.isfinite(sigma_initial) or sigma_initial <= 0:
        return {
            "valid_window": False,
            "n_events": 0,
            "noise_sigma_20hz_hp_dff": np.nan,
            "peak_amp_median_dff": np.nan,
            "peak_amp_mean_dff": np.nan,
            "peak_amp_p90_dff": np.nan,
            "snr_peak_median": np.nan,
            "snr_peak_mean": np.nan,
            "snr_peak_p90": np.nan,
            "event_rate_hz": np.nan,
            "event_times_sec": np.array([], dtype=float),
            "event_peak_amplitudes": np.array([], dtype=float),
            "trace_corrected": y,
            "trace_time_sec": t,
            "event_mask": np.zeros_like(y, dtype=bool),
        }

    min_dist = max(1, int(round(MIN_PEAK_DISTANCE_MS * 1e-3 * fs)))
    peaks, props = find_peaks(
        y_det,
        height=EVENT_THRESHOLD_SD * sigma_initial,
        prominence=EVENT_PROMINENCE_SD * sigma_initial,
        distance=min_dist,
    )

    pre = int(round(EVENT_MASK_PRE_MS * 1e-3 * fs))
    post = int(round(EVENT_MASK_POST_MS * 1e-3 * fs))
    event_mask = np.zeros_like(y_det, dtype=bool)
    for p in peaks:
        i0 = max(0, p - pre)
        i1 = min(len(event_mask), p + post + 1)
        event_mask[i0:i1] = True

    # Remove detected events before estimating noise, similar to spike-removed noise estimation.
    y_for_noise = y_det.copy()
    if event_mask.any() and (~event_mask).sum() >= max(MIN_VALID_SAMPLES // 2, 20):
        idx = np.arange(len(y_for_noise))
        y_for_noise[event_mask] = np.interp(idx[event_mask], idx[~event_mask], y_for_noise[~event_mask])

    hp_noise = highpass_filter(y_for_noise, fs, NOISE_HIGHPASS_HZ)
    noise_sigma = np.nanstd(hp_noise)

    peak_amps = y_det[peaks] if len(peaks) else np.array([], dtype=float)
    if peak_amps.size == 0 or not np.isfinite(noise_sigma) or noise_sigma <= 0:
        snrs = np.array([], dtype=float)
    else:
        snrs = peak_amps / noise_sigma

    duration = float(t[-1] - t[0]) if len(t) > 1 else np.nan

    return {
        "valid_window": bool(len(x) >= MIN_VALID_SAMPLES and np.isfinite(noise_sigma) and noise_sigma > 0),
        "n_events": int(len(peaks)),
        "noise_sigma_20hz_hp_dff": float(noise_sigma) if np.isfinite(noise_sigma) else np.nan,
        "peak_amp_median_dff": float(np.nanmedian(peak_amps)) if peak_amps.size else np.nan,
        "peak_amp_mean_dff": float(np.nanmean(peak_amps)) if peak_amps.size else np.nan,
        "peak_amp_p90_dff": float(np.nanpercentile(peak_amps, 90)) if peak_amps.size else np.nan,
        "snr_peak_median": float(np.nanmedian(snrs)) if snrs.size else np.nan,
        "snr_peak_mean": float(np.nanmean(snrs)) if snrs.size else np.nan,
        "snr_peak_p90": float(np.nanpercentile(snrs, 90)) if snrs.size else np.nan,
        "event_rate_hz": float(len(peaks) / duration) if np.isfinite(duration) and duration > 0 else np.nan,
        "event_times_sec": t[peaks] if len(peaks) else np.array([], dtype=float),
        "event_peak_amplitudes": peak_amps,
        "trace_corrected": y,
        "trace_time_sec": t,
        "event_mask": event_mask,
    }

## Example session: compute ROI metrics and cache a trace segment

In [ ]:
example_asset = assets[EXAMPLE_ASSET_INDEX]
example_h5 = resolve_trace_h5(example_asset)
example_session_id = get_asset_session_id(example_asset)

print(f"Example session: {example_session_id}")
print(example_h5)

example_rows = []
example_trace_cache = {}

for dmd in ["DMD1", "DMD2"]:
    info = get_dmd_info(example_h5, dmd)
    t0 = info["t_start"] + 30.0 if TRACE_START_SEC is None else float(TRACE_START_SEC)
    t1 = t0 + float(TRACE_DURATION_SEC)
    t_seg, X_seg = read_dmd_segment_all_rois(example_h5, dmd, t0, t1)

    for roi_idx, roi_id in enumerate(info["roi_ids"]):
        x = X_seg[roi_idx]
        res = compute_hao_like_snr(x, t_seg)
        example_rows.append({
            "session_id": example_session_id,
            "asset_index": EXAMPLE_ASSET_INDEX,
            "dmd": dmd,
            "roi_index": roi_idx,
            "roi_id": roi_id,
            "depth_um": get_asset_depth(example_asset, dmd),
            "n_events": res["n_events"],
            "event_rate_hz": res["event_rate_hz"],
            "noise_sigma_20hz_hp_dff": res["noise_sigma_20hz_hp_dff"],
            "peak_amp_median_dff": res["peak_amp_median_dff"],
            "peak_amp_mean_dff": res["peak_amp_mean_dff"],
            "peak_amp_p90_dff": res["peak_amp_p90_dff"],
            "snr_peak_median": res["snr_peak_median"],
            "snr_peak_mean": res["snr_peak_mean"],
            "snr_peak_p90": res["snr_peak_p90"],
            "valid_roi": bool(res["n_events"] >= MIN_EVENTS_FOR_VALID_ROI and np.isfinite(res[SNR_METRIC])),
        })
        example_trace_cache[(dmd, roi_id)] = {
            "t": res["trace_time_sec"],
            "x_corrected": res["trace_corrected"],
            "event_times": res["event_times_sec"],
            "event_peak_amplitudes": res["event_peak_amplitudes"],
            "event_mask": res["event_mask"],
        }

example_snr_df = pd.DataFrame(example_rows)
valid_example_snr_df = example_snr_df[example_snr_df["valid_roi"]].copy()

print(f"Example valid ROIs: {len(valid_example_snr_df)} / {len(example_snr_df)}")
display(example_snr_df.sort_values(SNR_METRIC, ascending=False).head(10))

## Example-session traces sorted bySNR

This cell intentionally contains all plotting code inline. Edit directly here.

In [ ]:
df = valid_example_snr_df.sort_values(SNR_METRIC, ascending=False).copy()

# Plot all valid ROIs, sorted from high to low SNR. Increase/decrease this if the plot becomes too tall.
MAX_ROIS_TO_PLOT = 10
VERTICAL_SPACING = 3.0
LINEWIDTH = 0.8

plot_df = df.head(MAX_ROIS_TO_PLOT).reset_index(drop=True)

fig_height = max(5, 0.28 * len(plot_df) + 1.8)
fig, ax = plt.subplots(figsize=(12, fig_height))

for i, row in plot_df.iterrows():
    cache = example_trace_cache[(row["dmd"], row["roi_id"])]
    t = cache["t"]
    x = cache["x_corrected"]
    if len(t) == 0 or len(x) == 0:
        continue
    tt = t - t[0]
    # Robustly scale each trace for stacked visualization, but annotate real SNR.
    scale = np.nanpercentile(np.abs(x), 95)
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0
    yy = x / scale + i * VERTICAL_SPACING
    ax.plot(tt, yy, lw=LINEWIDTH)

    ev = cache["event_times"]
    if len(ev):
        ev_tt = ev - t[0]
        ev_tt = ev_tt[(ev_tt >= tt[0]) & (ev_tt <= tt[-1])]
#         ax.plot(ev_tt, np.full_like(ev_tt, i * VERTICAL_SPACING + 1.15), "|", ms=7, alpha=0.7)

    ax.text(
        tt[-1] + 0.05,
        i * VERTICAL_SPACING,
        f'{row["dmd"]} ROI {row["roi_id"].split("0")[-1]}  SNR={row[SNR_METRIC]:.1f}',
        va="center",
        fontsize=8,
    )

ax.set_xlabel("Time in example window (s)")
ax.set_ylabel("ROIs (high \u2192 low SNR)")
ax.set_title(f"Example session {example_session_id}: baseline-corrected traces sorted by {SNR_METRIC}")
ax.set_yticks([])
ax.set_xlim(-0.1, TRACE_DURATION_SEC + 3.0)
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for spine in ['left','bottom','left','right']:
    ax.spines[spine].set_linewidth(2)
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f"{today_str}_ASAP7_HaoSNR_example_traces_sorted.png", dpi=300, bbox_inches="tight")
plt.show()

## Selected high / medium / low SNR traces with event markers

In [ ]:
df = valid_example_snr_df.sort_values(SNR_METRIC, ascending=False).copy().reset_index(drop=True)

if len(df) >= 3:
    pick_indices = [0, len(df) // 2, len(df) - 1]
else:
    pick_indices = list(range(len(df)))

fig, axes = plt.subplots(len(pick_indices), 1, figsize=(12, 2.6 * len(pick_indices)), sharex=True)
if len(pick_indices) == 1:
    axes = [axes]

for ax, idx in zip(axes, pick_indices):
    row = df.iloc[idx]
    cache = example_trace_cache[(row["dmd"], row["roi_id"])]
    t = cache["t"]
    x = cache["x_corrected"]
    tt = t - t[0]
    ax.plot(tt, x, lw=0.9)
    ax.axhline(0, lw=0.8, color="0.5", alpha=0.5)

    ev = cache["event_times"]
    ev_amp = cache["event_peak_amplitudes"]
    if len(ev):
        ev_tt = ev - t[0]
        in_view = (ev_tt >= tt[0]) & (ev_tt <= tt[-1])
        ev_tt = ev_tt[in_view]
        ev_amp = ev_amp[in_view]
        ax.scatter(ev_tt, ev_amp, s=20, zorder=3, label="detected peaks")

    ax.set_ylabel("baseline-corrected\nΔF/F")
    ax.set_title(
        f'{row["dmd"]} {row["roi_id"]} | {SNR_METRIC}={row[SNR_METRIC]:.2f} | '
        f'noise={row["noise_sigma_20hz_hp_dff"]:.4g} | median peak={row["peak_amp_median_dff"]:.4g}'
    )
    ax.legend(frameon=False, loc="upper right")

axes[-1].set_xlabel("Time in example window (s)")
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f"{today_str}_ASAP7_HaoSNR_example_high_mid_low.png", dpi=300, bbox_inches="tight")
plt.show()

## Compute all-session ROI-level Hao-like SNR

This loops through each registry session, reads distributed windows from each DMD, computes the Hao-like SNR metric per ROI per window, then aggregates to one row per ROI.

In [ ]:
window_rows = []
errors = []

for asset_index, asset in enumerate(assets):
    session_id = get_asset_session_id(asset)
    subject_id = get_asset_subject_id(asset)

    try:
        h5_path = resolve_trace_h5(asset)
    except Exception as exc:
        errors.append({"asset_index": asset_index, "session_id": session_id, "error": repr(exc)})
        print(f"SKIP {session_id}: {exc}")
        continue

    print(f"\n[{asset_index + 1}/{len(assets)}] {session_id}")
    print(h5_path)

    for dmd in ["DMD1", "DMD2"]:
        try:
            info = get_dmd_info(h5_path, dmd)
        except Exception as exc:
            errors.append({"asset_index": asset_index, "session_id": session_id, "dmd": dmd, "error": repr(exc)})
            print(f"  SKIP {dmd}: {exc}")
            continue

        t_start = info["t_start"]
        t_stop = info["t_stop"]
        duration = t_stop - t_start
        if duration < METRIC_WINDOW_SEC:
            print(f"  SKIP {dmd}: duration too short ({duration:.1f}s)")
            continue

        starts = np.linspace(t_start, t_stop - METRIC_WINDOW_SEC, N_METRIC_WINDOWS)
        print(f"  {dmd}: {info['n_rois']} ROIs, {len(starts)} windows, fs≈{info['fs']:.1f} Hz")

        for win_index, w0 in enumerate(starts):
            w1 = w0 + METRIC_WINDOW_SEC
            try:
                t_seg, X_seg = read_dmd_segment_all_rois(h5_path, dmd, w0, w1)
            except Exception as exc:
                errors.append({
                    "asset_index": asset_index,
                    "session_id": session_id,
                    "dmd": dmd,
                    "window_index": win_index,
                    "error": repr(exc),
                })
                continue

            for roi_idx, roi_id in enumerate(info["roi_ids"]):
                res = compute_hao_like_snr(X_seg[roi_idx], t_seg)
                window_rows.append({
                    "asset_index": asset_index,
                    "session_id": session_id,
                    "subject_id": subject_id,
                    "dmd": dmd,
                    "depth_um": get_asset_depth(asset, dmd),
                    "roi_index": roi_idx,
                    "roi_id": roi_id,
                    "window_index": win_index,
                    "window_start_sec": float(w0),
                    "window_stop_sec": float(w1),
                    "valid_window": res["valid_window"],
                    "n_events": res["n_events"],
                    "event_rate_hz": res["event_rate_hz"],
                    "noise_sigma_20hz_hp_dff": res["noise_sigma_20hz_hp_dff"],
                    "peak_amp_median_dff": res["peak_amp_median_dff"],
                    "peak_amp_mean_dff": res["peak_amp_mean_dff"],
                    "peak_amp_p90_dff": res["peak_amp_p90_dff"],
                    "snr_peak_median": res["snr_peak_median"],
                    "snr_peak_mean": res["snr_peak_mean"],
                    "snr_peak_p90": res["snr_peak_p90"],
                })

window_snr_df = pd.DataFrame(window_rows)
error_df = pd.DataFrame(errors)

print(f"\nComputed {len(window_snr_df)} ROI-window rows")
if len(error_df):
    print(f"Encountered {len(error_df)} errors/skipped chunks")
    display(error_df.head(20))

display(window_snr_df.head())

In [ ]:
# Aggregate window-level metrics to one row per ROI.
# For amplitude and SNR, median across windows is the most robust summary.
# Noise is also summarized by median across windows.

if len(window_snr_df) == 0:
    raise RuntimeError("No window-level SNR rows were computed.")

group_cols = ["asset_index", "session_id", "subject_id", "dmd", "depth_um", "roi_index", "roi_id"]

roi_summary_df = (
    window_snr_df
    .groupby(group_cols, dropna=False)
    .agg(
        n_valid_windows=("valid_window", "sum"),
        n_total_windows=("valid_window", "size"),
        n_events_total=("n_events", "sum"),
        event_rate_hz_median=("event_rate_hz", "median"),
        noise_sigma_20hz_hp_dff=("noise_sigma_20hz_hp_dff", "median"),
        peak_amp_median_dff=("peak_amp_median_dff", "median"),
        peak_amp_mean_dff=("peak_amp_mean_dff", "median"),
        peak_amp_p90_dff=("peak_amp_p90_dff", "median"),
        snr_peak_median=("snr_peak_median", "median"),
        snr_peak_mean=("snr_peak_mean", "median"),
        snr_peak_p90=("snr_peak_p90", "median"),
    )
    .reset_index()
)

roi_summary_df["valid_roi"] = (
    (roi_summary_df["n_valid_windows"] >= MIN_VALID_WINDOWS) &
    (roi_summary_df["n_events_total"] >= MIN_EVENTS_FOR_VALID_ROI) &
    np.isfinite(roi_summary_df[SNR_METRIC]) &
    (roi_summary_df[SNR_METRIC] > 0)
)

valid_snr_df = roi_summary_df[roi_summary_df["valid_roi"]].copy()

print(f"Valid ROIs: {len(valid_snr_df)} / {len(roi_summary_df)}")
display(roi_summary_df.sort_values(SNR_METRIC, ascending=False).head(15))

if SAVE_TABLE:
    window_csv = SAVE_PATH / f"{today_str}_ASAP7_HaoSNR_window_metrics.csv"
    roi_csv = SAVE_PATH / f"{today_str}_ASAP7_HaoSNR_roi_summary.csv"
    window_snr_df.to_csv(window_csv, index=False)
    roi_summary_df.to_csv(roi_csv, index=False)
    print(f"Saved window metrics: {window_csv}")
    print(f"Saved ROI summary: {roi_csv}")

## Pooled SNR histograms

In [ ]:
df = valid_snr_df.copy()
metric = SNR_METRIC

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax in axes.flatten():
    ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
    ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
    for spine in ['left','bottom','top','right']:
        ax.spines[spine].set_linewidth(2)

plot_specs = [
    ("All ROIs", df),
    ("DMD1", df[df["dmd"] == "DMD1"]),
    ("DMD2", df[df["dmd"] == "DMD2"]),
]

# Use shared bins for all panels.
vals_all = df[metric].replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
if vals_all.size:
    lo, hi = np.nanpercentile(vals_all, [1, 99])
    if lo == hi:
        lo, hi = np.nanmin(vals_all), np.nanmax(vals_all)
    bins = np.linspace(max(0, lo), hi, HIST_BINS)
else:
    bins = HIST_BINS

for ax, (title, sub) in zip(axes, plot_specs):
    vals = sub[metric].replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
    ax.hist(vals, bins=bins, alpha=0.8, edgecolor="white")
    if vals.size:
        ax.axvline(np.nanmedian(vals), lw=2, ls="--", label=f"median={np.nanmedian(vals):.2f}",color='k')
    ax.set_title(f"{title}\nN={len(vals)} ROIs")
    ax.set_xlabel("SNR")
    ax.legend(frameon=False)

axes[0].set_ylabel("ROI count")
fig.suptitle("ASAP7y ROI SNR distributions",fontsize=18)
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f"{today_str}_ASAP7_HaoSNR_histograms.png", dpi=300, bbox_inches="tight")
plt.show()

## Plot 4: event signal amplitude versus 20 Hz high-pass noise

In [ ]:
df = valid_snr_df.copy()

xcol = "noise_sigma_20hz_hp_dff"
ycol = "peak_amp_median_dff"

fig, ax = plt.subplots(figsize=(7, 6))

for dmd, sub in df.groupby("dmd", sort=True):
    x = sub[xcol].to_numpy(dtype=float)
    y = sub[ycol].to_numpy(dtype=float)
    keep = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    ax.scatter(x[keep], y[keep], s=28, alpha=SCATTER_ALPHA, label=f"{dmd} (n={keep.sum()})")

all_x = df[xcol].to_numpy(dtype=float)
all_x = all_x[np.isfinite(all_x) & (all_x > 0)]
if all_x.size:
    xmin, xmax = np.nanpercentile(all_x, [1, 99])
    if xmin <= 0 or xmin == xmax:
        xmin, xmax = np.nanmin(all_x), np.nanmax(all_x)
    xx = np.geomspace(xmin, xmax, 200)
    for snr in [1, 2, 5, 10, 20]:
        ax.plot(xx, snr * xx, ls="--", lw=0.9, color="0.35")
        ax.text(xx[-1], snr * xx[-1], f" SNR={snr}", va="center", fontsize=8, color="0.25")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"Noise: SD of 20 Hz high-pass, event-removed trace ($\Delta F/F$)")
ax.set_ylabel(r"Signal: median detected event peak ($\Delta F/F$)")
ax.set_title("Hao-like ASAP7y event amplitude versus noise")
ax.legend(frameon=False)
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f"{today_str}_ASAP7_HaoSNR_amplitude_vs_noise.png", dpi=300, bbox_inches="tight")
plt.show()

## Plot 5: DMD/depth comparison of signal, noise, SNR, and event rate

In [ ]:
df = valid_snr_df.copy()
metrics = [
    ("peak_amp_median_dff", "Median event peak\nΔF/F"),
    ("noise_sigma_20hz_hp_dff", "20 Hz HP event-removed\nnoise SD"),
    (SNR_METRIC, "Hao-like\nevent SNR"),
    ("event_rate_hz_median", "Median event rate\nHz"),
]

fig, axes = plt.subplots(1, len(metrics), figsize=(4.0 * len(metrics), 4.5))
if len(metrics) == 1:
    axes = [axes]

rng = np.random.default_rng(0)
for ax, (metric, ylabel) in zip(axes, metrics):
    groups = []
    labels = []
    for dmd, sub in df.groupby("dmd", sort=True):
        vals = sub[metric].replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)
        groups.append(vals)
        labels.append(dmd)

    ax.boxplot(groups, labels=labels, showfliers=False)
    for i, vals in enumerate(groups, start=1):
        jitter = rng.normal(0, 0.045, size=len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals, s=16, alpha=0.35)

    ax.set_ylabel(ylabel)
    ax.set_title(metric)

fig.suptitle("ASAP7y Hao-like SNR components by DMD")
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f"{today_str}_ASAP7_HaoSNR_by_DMD.png", dpi=300, bbox_inches="tight")
plt.show()

## Quick DMD1 vs DMD2 tests

This is useful for slide annotation, but treat it as descriptive because ROIs are nested within sessions/mice. For a stronger claim, use the session-level table below or a hierarchical model.

In [ ]:
metrics_to_test = [
    "peak_amp_median_dff",
    "noise_sigma_20hz_hp_dff",
    SNR_METRIC,
    "event_rate_hz_median",
]

rows = []
print("Mann-Whitney U tests: DMD1 vs DMD2")
print("=" * 90)

for metric in metrics_to_test:
    x1 = valid_snr_df.loc[valid_snr_df["dmd"] == "DMD1", metric].replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)
    x2 = valid_snr_df.loc[valid_snr_df["dmd"] == "DMD2", metric].replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)
    if len(x1) == 0 or len(x2) == 0:
        continue
    stat, p = mannwhitneyu(x1, x2, alternative="two-sided")
    med1 = np.nanmedian(x1)
    med2 = np.nanmedian(x2)
    frac_delta = (med2 - med1) / med1 if med1 != 0 else np.nan
    rows.append({
        "metric": metric,
        "n_dmd1": len(x1),
        "n_dmd2": len(x2),
        "median_dmd1": med1,
        "median_dmd2": med2,
        "frac_delta_dmd2_vs_dmd1": frac_delta,
        "mannwhitney_U": stat,
        "p_value": p,
    })

roi_dmd_test_df = pd.DataFrame(rows)

if len(roi_dmd_test_df) and HAS_STATSMODELS:
    reject, p_fdr, _, _ = multipletests(roi_dmd_test_df["p_value"].to_numpy(), alpha=0.05, method="fdr_bh")
    roi_dmd_test_df["p_fdr_bh"] = p_fdr
    roi_dmd_test_df["significant_fdr_0p05"] = reject

for _, row in roi_dmd_test_df.iterrows():
    if "p_fdr_bh" in roi_dmd_test_df.columns:
        print(
            f"{row['metric']:28s} | "
            f"DMD1 med={row['median_dmd1']:.4g}, DMD2 med={row['median_dmd2']:.4g}, "
            f"frac Δ={row['frac_delta_dmd2_vs_dmd1']:+.1%}, "
            f"raw p={row['p_value']:.3e}, FDR p={row['p_fdr_bh']:.3e}"
        )
    else:
        print(
            f"{row['metric']:28s} | "
            f"DMD1 med={row['median_dmd1']:.4g}, DMD2 med={row['median_dmd2']:.4g}, "
            f"frac Δ={row['frac_delta_dmd2_vs_dmd1']:+.1%}, "
            f"p={row['p_value']:.3e}"
        )

display(roi_dmd_test_df)

## Session-level summaries

Use these for more conservative depth/DMD comparisons because they reduce ROI-level pseudoreplication.

In [ ]:
session_summary_df = (
    valid_snr_df
    .groupby(["session_id", "subject_id", "dmd", "depth_um"], dropna=False)
    .agg(
        n_rois=("roi_id", "size"),
        peak_amp_median_dff=("peak_amp_median_dff", "median"),
        noise_sigma_20hz_hp_dff=("noise_sigma_20hz_hp_dff", "median"),
        snr_peak_median=("snr_peak_median", "median"),
        snr_peak_mean=("snr_peak_mean", "median"),
        snr_peak_p90=("snr_peak_p90", "median"),
        event_rate_hz_median=("event_rate_hz_median", "median"),
    )
    .reset_index()
)

display(session_summary_df)

if SAVE_TABLE:
    session_csv = SAVE_PATH / f"{today_str}_ASAP7_HaoSNR_session_summary.csv"
    session_summary_df.to_csv(session_csv, index=False)
    print(f"Saved session summary: {session_csv}")

In [ ]:
# Session-level paired-ish view by session and DMD.
# This plot is more conservative than the ROI-level plot because each point is a session × DMD summary.

df = session_summary_df.copy()
metric = SNR_METRIC

fig, ax = plt.subplots(figsize=(5.5, 4.5))

rng = np.random.default_rng(1)
for i, dmd in enumerate(["DMD1", "DMD2"], start=1):
    vals = df.loc[df["dmd"] == dmd, metric].replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)
    if len(vals):
        ax.boxplot([vals], positions=[i], widths=0.45, showfliers=False)
        ax.scatter(np.full(len(vals), i) + rng.normal(0, 0.04, len(vals)), vals, s=36, alpha=0.75, label=dmd)

# Draw faint lines for sessions that have both DMDs.
wide = df.pivot_table(index="session_id", columns="dmd", values=metric, aggfunc="median")
for session_id, row in wide.iterrows():
    if np.isfinite(row.get("DMD1", np.nan)) and np.isfinite(row.get("DMD2", np.nan)):
        ax.plot([1, 2], [row["DMD1"], row["DMD2"]], color="0.6", alpha=0.35, lw=1)

ax.set_xticks([1, 2])
ax.set_xticklabels(["DMD1", "DMD2"])
ax.set_ylabel("Session median Hao-like event SNR")
ax.set_title("Session-level SNR by DMD")
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f"{today_str}_ASAP7_HaoSNR_session_by_DMD.png", dpi=300, bbox_inches="tight")
plt.show()

## Interpretation notes

- This metric should be easier to explain than the previous dynamic-range or biological-MAD metrics: **detected event peak divided by high-frequency event-removed noise**.
- It follows the same practical SNR convention used in ASAP5 benchmarking: event signal peak over the standard deviation of a 20 Hz high-pass filtered trace after removing spikes/events.
- Because the denominator explicitly excludes slower subthreshold/task-band structure, this metric is best interpreted as **event detectability**, not the total variance or task-response strength of each ROI.
- If the event detector misses true voltage events, the signal numerator will be underestimated. If it detects artifacts, the numerator will be inflated.
- ROI-level comparisons are descriptive because ROIs are nested within sessions and mice. Use the session-level summary for conservative DMD/depth claims.